<a href="https://colab.research.google.com/github/dlwnsgur4378/-Vit-/blob/main/pet_ai4_FINAL_336_5crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pet-AI4 FINAL — 논문형 baseline + HuFEP ViT-H/14 + 336 + 5Crop 동시학습

이 노트북은 업로드한 HuFEP 논문 구조를 기준으로, 다음 목표에 맞게 수정한 최종형입니다.

- 논문 구조 유지: **baseline ViT 모델** + **HuFEP ViT 모델** 2개를 각각 학습하고 비교
- 논문과 같은 얼굴 중심 전처리 철학 유지: `crop_headsafe`, `crop_tight` 우선
- 실전 성능 보강: `crop_yolo`, `crop_center`, `crop_original`까지 포함
- 학습 방식 변경: 이미지 1장당 5개 crop을 **동시에 모델에 넣고 가중 평균 logits로 loss 계산**
- 샘플 수는 train 이미지 수 기준 유지: 3200장이라면 dataset 길이도 약 3200장
- 입력 크기: `IMG_SIZE = 336`  
  - 224보다 얼굴 디테일을 더 봄
  - 518보다 Colab에서 훨씬 현실적
- 최종 평가: baseline 단독, HuFEP 단독, baseline+HuFEP ensemble 비교

> 실행 순서: 0번부터 마지막까지 순서대로 실행하면 됩니다.  
> GPU 메모리 부족이 뜨면 3번 설정 칸에서 `IMG_SIZE = 224`로 낮추고 다시 실행하세요.


In [ ]:
# =========================================================
# 0. Google Drive 마운트
# =========================================================
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# =========================================================
# 1. pet-ai4(3) 경로 / 파일 확인
# =========================================================
from pathlib import Path
import os, glob, json, shutil, textwrap

PROJECT_BASE = Path("/content/drive/MyDrive/pet-ai4(3)")
PROJECT_BASE.mkdir(parents=True, exist_ok=True)

DOG_DATASET_ZIP = PROJECT_BASE / "dataset.zip"
HUMAN_ARCHIVE_ZIP = PROJECT_BASE / "archive.zip"
DIMA806_DIR = PROJECT_BASE / "hufep_dima806"  # 이번 버전은 필수 아님

PATHS = {
    "dog_dataset_zip 필수": DOG_DATASET_ZIP,
    "human_archive_zip 필수": HUMAN_ARCHIVE_ZIP,
    "dima806_dir 선택": DIMA806_DIR,
    "optional_yolo26x_detector": PROJECT_BASE / "yolo26x.pt",
    "optional_yolo26n_detector": PROJECT_BASE / "yolo26n.pt",
}

print("PROJECT_BASE:", PROJECT_BASE)
for k, p in PATHS.items():
    print(f"{k:32s}: {'✅' if p.exists() else '❌'} {p}")

print("\n현재 pet-ai4(3) 폴더 내용:")
!find "{PROJECT_BASE}" -maxdepth 3 -print | sed -n '1,180p'

if not DOG_DATASET_ZIP.exists():
    print("\n❌ dataset.zip이 없습니다. /content/drive/MyDrive/pet-ai4(3)/dataset.zip 위치에 올려주세요.")
else:
    print("\n✅ dataset.zip 확인")

if not HUMAN_ARCHIVE_ZIP.exists():
    print("\n❌ archive.zip이 없습니다. 새 HuFEP ViT-H/14를 만들려면 archive.zip이 필요합니다.")
else:
    print("✅ archive.zip 확인 — 이것으로 새 HuFEP ViT-H/14를 먼저 만듭니다.")

print("\n참고: hufep_dima806는 이번 버전에서 필수가 아닙니다.")


PROJECT_BASE: /content/drive/MyDrive/pet-ai4(3)
dog_dataset_zip 필수              : ❌ /content/drive/MyDrive/pet-ai4(3)/dataset.zip
human_archive_zip 필수            : ❌ /content/drive/MyDrive/pet-ai4(3)/archive.zip
dima806_dir 선택                  : ❌ /content/drive/MyDrive/pet-ai4(3)/hufep_dima806
optional_yolo26x_detector       : ❌ /content/drive/MyDrive/pet-ai4(3)/yolo26x.pt
optional_yolo26n_detector       : ❌ /content/drive/MyDrive/pet-ai4(3)/yolo26n.pt

현재 pet-ai4(3) 폴더 내용:
/content/drive/MyDrive/pet-ai4(3)

❌ dataset.zip이 없습니다. /content/drive/MyDrive/pet-ai4(3)/dataset.zip 위치에 올려주세요.

❌ archive.zip이 없습니다. 새 HuFEP ViT-H/14를 만들려면 archive.zip이 필요합니다.

참고: hufep_dima806는 이번 버전에서 필수가 아닙니다.


In [ ]:

# =========================================================
# 2. 패키지 설치 — Colab pandas 충돌 방지 + YOLO26 지원
# =========================================================

# pandas는 Colab 호환 때문에 -U 대상에서 제외합니다.
!pip -q install -U timm transformers safetensors accelerate ultralytics scikit-learn matplotlib tqdm opencv-python-headless
!pip -q install "pandas==2.2.2"

import pandas as pd
import ultralytics
print("✅ pandas version:", pd.__version__)
print("✅ ultralytics version:", ultralytics.__version__)

if pd.__version__ != "2.2.2":
    print("⚠️ pandas가 2.2.2가 아닙니다. 런타임 다시 시작 후 0번부터 다시 실행하세요.")
else:
    print("✅ pandas 충돌 방지 완료")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 98.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ pandas version: 2.2.2
✅ ultralytics version: 8.4.48
✅ pandas 충돌 방지 완료


In [ ]:

# =========================================================
# 3. 공통 설정 — FINAL 336 + 5CROP SIMULTANEOUS + baseline/HuFEP
# - 논문형 구조: baseline ViT vs HuFEP ViT 2모델 비교
# - 입력 크기: 224와 518 사이의 균형값인 336
# - 학습: 이미지 1장당 5개 crop을 동시에 넣고 가중 평균 logits로 학습
# - dataset 길이는 train 원본 이미지 수 기준으로 유지
# =========================================================

import os, json, math, time, random, zipfile, shutil, gc, hashlib, warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageOps, ImageDraw
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as TVT

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import timm
from timm.data import resolve_model_data_config, create_transform

# -------------------------
# 경로
# -------------------------
PROJECT_BASE = Path("/content/drive/MyDrive/pet-ai4(3)")
DOG_DATASET_ZIP = PROJECT_BASE / "dataset.zip"
HUMAN_ARCHIVE_ZIP = PROJECT_BASE / "archive.zip"

# 기존 224 실험 결과. human HuFEP pretrain source로 사용.
OLD_OUTPUT_DIR = PROJECT_BASE / "output_v4_ARCHIVE_HuFEP_ViTH14_BALANCED_V3MULTICROP"
OLD_HUMAN_PRETRAIN_DIR = OLD_OUTPUT_DIR / "00_archive_HuFEP_ViTH14_human_pretrain"
OLD_HUFEP_DOG_DIR = OLD_OUTPUT_DIR / "02_HuFEP_ViTH14_archive_then_dog_face"

# 새 FINAL 336 + 5crop 동시학습 결과 폴더
OUTPUT_DIR = PROJECT_BASE / "output_v4_FINAL_PAPERLIKE_ViTH14_336_5CROP_SIMUL_BASELINE_HUFEP"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WORK_DIR = Path("/content/pet_ai4_final_vith14_336_5crop_simul_work")
DOG_EXTRACT_DIR = WORK_DIR / "dog_dataset"
HUMAN_EXTRACT_DIR = WORK_DIR / "human_archive"

# crop cache는 IMG_SIZE와 무관한 이미지 crop 파일이므로 기존 1280 cache를 재사용
CROP_CACHE_DIR = Path("/content/pet_ai4_archive_hufep_vith14_balanced_work/yolo26x_crop_cache_1280")

# -------------------------
# 클래스
# -------------------------
DOG_CLASSES = ["angry", "happy", "relaxed", "sad"]
DOG_CLASS_TO_IDX = {c: i for i, c in enumerate(DOG_CLASSES)}
DOG_IDX_TO_CLASS = {i: c for c, i in DOG_CLASS_TO_IDX.items()}

HUMAN_CLASSES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
HUMAN_CLASS_TO_IDX = {c: i for i, c in enumerate(HUMAN_CLASSES)}
HUMAN_IDX_TO_CLASS = {i: c for c, i in HUMAN_CLASS_TO_IDX.items()}

# -------------------------
# FINAL 336 설정
# -------------------------
SEED = 42
IMG_SIZE = 336

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_WORKERS = 0
USE_AMP = True

# V3처럼 낮추지 않음
MAX_EPOCHS_DOG = 70
EARLY_STOP_PATIENCE_DOG = 14
HEAD_EPOCHS = 5

# archive human pretrain은 기존 224 checkpoint를 source로 사용. 없을 때만 20 epoch.
EPOCHS_ARCHIVE_HUMAN_PRETRAIN = 20
HUMAN_PRETRAIN_PATIENCE = 10
HUMAN_PRETRAIN_SWITCH_FULL_EPOCH = 2

# baseline V3 LR
LR_HEAD = 1e-4
LR_BACKBONE = 2e-5
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.06
GRAD_CLIP_NORM = 1.0

# HuFEP도 낮추지 않고 V3값 사용
LR_HEAD_HUFEP = 1e-4
LR_BACKBONE_HUFEP = 2e-5
LR_FINAL_UNFREEZE_HUFEP = 2e-5
FREEZE_RATIO_HUFEP = 0.0
MAX_EPOCHS_DOG_HUFEP = 70
EARLY_STOP_PATIENCE_DOG_HUFEP = 14
HEAD_EPOCHS_HUFEP = 5
UNFREEZE_EPOCH_HUFEP = 5
USE_COSINE_HUFEP = True
MIN_LR_HUFEP = 1e-7

# split: 80 / 10 / 10
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# -------------------------
# YOLO crop 설정
# -------------------------
DETECTOR_PREFERENCE = "yolo26x"
DETECT_CONF = 0.25
DETECT_IOU = 0.50
DETECT_IMGSZ = 1280
MIN_BOX_AREA_RATIO = 0.010
CROP_PADDING = 0.20
TIGHT_PADDING = 0.08
HEAD_TOP_RATIO = 0.60

# -------------------------
# HuFEP crop 설정
# -------------------------
# 5개 crop 후보를 모두 사용합니다.
# 단, 5개를 샘플로 펼치는 것이 아니라 이미지 1장 안에 [5, C, H, W]로 묶어서 동시에 학습합니다.
# train 이미지가 약 3200장이면 dataset 길이도 약 3200입니다.
HUFEP_RANDOM_ONE_CROP_PER_IMAGE = False
BASELINE_RANDOM_ONE_CROP_PER_IMAGE = False
USE_5CROP_SIMULTANEOUS_TRAIN = True
USE_5CROP_SIMULTANEOUS_VAL = True
HUFEP_DETECTED_ONLY_TRAIN = False
HUFEP_MIN_DET_CONF = 0.0
HUFEP_MIN_CLEAN_PER_CLASS = 200

ALL_DOG_CROP_COLS_5 = [
    "crop_headsafe",
    "crop_tight",
    "crop_yolo",
    "crop_center",
    "crop_original",
]

# 얼굴 crop을 더 중요하게 보는 논문형 가중치.
# 학습/검증/추론에서 5개 crop logits/probability 평균에 사용합니다.
CROP_WEIGHTS = {
    "crop_headsafe": 0.35,
    "crop_tight": 0.25,
    "crop_yolo": 0.20,
    "crop_center": 0.10,
    "crop_original": 0.10,
}
MULTICROP_WEIGHT_LIST = [float(CROP_WEIGHTS[c]) for c in ALL_DOG_CROP_COLS_5]
MULTICROP_WEIGHT_LIST = [w / sum(MULTICROP_WEIGHT_LIST) for w in MULTICROP_WEIGHT_LIST]

HUFEP_VITH14_DOG_TRAIN_CROP_COLS = ALL_DOG_CROP_COLS_5[:]
HUFEP_DETECTED_TRAIN_CROP_COLS = HUFEP_VITH14_DOG_TRAIN_CROP_COLS[:]
HUFEP_FALLBACK_TRAIN_CROP_COLS = ALL_DOG_CROP_COLS_5[:]
HUFEP_EVAL_CROP_COL = "crop_headsafe"
HUFEP_USE_WEIGHTED_SAMPLER = True

# -------------------------
# Baseline: 5 crop random-one
# -------------------------
TIMM_TRAIN_CROP_COLS = ALL_DOG_CROP_COLS_5[:]
EXPAND_TRAIN_CROPS = False  # 5crop simultaneous dataset을 쓰므로 full expand 금지

# -------------------------
# 사람 표정 pretrain imbalance 보정
# -------------------------
HUMAN_USE_WEIGHTED_SAMPLER = True
HUMAN_USE_CLASS_WEIGHT_LOSS = True
HUMAN_CLASS_WEIGHT_POWER = 0.5
HUMAN_SAMPLER_WEIGHT_POWER = 1.0
HUMAN_CLASS_WEIGHT_CLIP_MAX = 4.0

# -------------------------
# 전처리
# -------------------------
HF_FORCE_PAPER_NORMALIZE = False
os.environ["HF_FORCE_PAPER_NORMALIZE"] = "0"
ENSEMBLE_STEP = 0.0125
VITH14_MODEL_NAME = "vit_huge_patch14_224.orig_in21k"

# -------------------------
# 출력 폴더
# -------------------------
HUMAN_PRETRAIN_DIR = OUTPUT_DIR / "00_archive_HuFEP_ViTH14_human_pretrain_336"
BASELINE_DIR = OUTPUT_DIR / "01_baseline_ViTH14_dog_only_336_5crop_simul"
HUFEP_DOG_STABLE_DIR = OUTPUT_DIR / "02_HuFEP_ViTH14_dog_336_5crop_simul"
HUFEP_DOG_DIR = HUFEP_DOG_STABLE_DIR  # 기존 코드 호환용

for d in [HUMAN_PRETRAIN_DIR, BASELINE_DIR, HUFEP_DOG_STABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -------------------------
# 호환용 alias
# -------------------------
MAX_EPOCHS = MAX_EPOCHS_DOG
EARLY_STOP_PATIENCE = EARLY_STOP_PATIENCE_DOG
EPOCHS_HUFEP_DOG = MAX_EPOCHS_DOG_HUFEP
EPOCHS_BASELINE_DOG = MAX_EPOCHS_DOG
EPOCHS_VITH14_DOG = MAX_EPOCHS_DOG

# -------------------------
# Device / Seed
# -------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("\n==============================")
print("✅ COMMON CONFIG LOADED — FINAL 336 + 5CROP SIMULTANEOUS + BASELINE/HUFEP")
print("==============================")
print("GPU:", GPU_NAME)
print("DEVICE:", DEVICE)
print("PROJECT_BASE:", PROJECT_BASE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("OLD_OUTPUT_DIR:", OLD_OUTPUT_DIR)
print("WORK_DIR:", WORK_DIR)
print("CROP_CACHE_DIR:", CROP_CACHE_DIR)
print("dataset.zip:", DOG_DATASET_ZIP.exists(), DOG_DATASET_ZIP)
print("archive.zip:", HUMAN_ARCHIVE_ZIP.exists(), HUMAN_ARCHIVE_ZIP)
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("MAX_EPOCHS_DOG:", MAX_EPOCHS_DOG)
print("EARLY_STOP_PATIENCE_DOG:", EARLY_STOP_PATIENCE_DOG)
print("LR_HEAD:", LR_HEAD)
print("LR_BACKBONE:", LR_BACKBONE)
print("LR_HEAD_HUFEP:", LR_HEAD_HUFEP)
print("LR_BACKBONE_HUFEP:", LR_BACKBONE_HUFEP)
print("FREEZE_RATIO_HUFEP:", FREEZE_RATIO_HUFEP)
print("HUFEP_RANDOM_ONE_CROP_PER_IMAGE:", HUFEP_RANDOM_ONE_CROP_PER_IMAGE)
print("HUFEP crop candidates:", HUFEP_VITH14_DOG_TRAIN_CROP_COLS)
print("baseline crops:", TIMM_TRAIN_CROP_COLS)
print("old human best:", (OLD_HUMAN_PRETRAIN_DIR / "best_model.pt").exists(), OLD_HUMAN_PRETRAIN_DIR / "best_model.pt")
print("old dog HuFEP best:", (OLD_HUFEP_DOG_DIR / "best_model.pt").exists(), OLD_HUFEP_DOG_DIR / "best_model.pt")
print("new human best:", (HUMAN_PRETRAIN_DIR / "best_model.pt").exists(), HUMAN_PRETRAIN_DIR / "best_model.pt")
print("336 HuFEP best:", (HUFEP_DOG_STABLE_DIR / "best_model.pt").exists(), HUFEP_DOG_STABLE_DIR / "best_model.pt")
print("336 baseline best:", (BASELINE_DIR / "best_model.pt").exists(), BASELINE_DIR / "best_model.pt")
print("USE_5CROP_SIMULTANEOUS_TRAIN:", USE_5CROP_SIMULTANEOUS_TRAIN)
print("CROP_WEIGHTS:", CROP_WEIGHTS)
print("==============================")



✅ COMMON CONFIG LOADED — FINAL 336 + 5CROP SIMULTANEOUS + BASELINE/HUFEP
GPU: Tesla T4
DEVICE: cuda
PROJECT_BASE: /content/drive/MyDrive/pet-ai4(3)
OUTPUT_DIR: /content/drive/MyDrive/pet-ai4(3)/output_v4_FINAL_PAPERLIKE_ViTH14_336_5CROP_SIMUL_BASELINE_HUFEP
OLD_OUTPUT_DIR: /content/drive/MyDrive/pet-ai4(3)/output_v4_ARCHIVE_HuFEP_ViTH14_BALANCED_V3MULTICROP
WORK_DIR: /content/pet_ai4_final_vith14_336_5crop_simul_work
CROP_CACHE_DIR: /content/pet_ai4_archive_hufep_vith14_balanced_work/yolo26x_crop_cache_1280
dataset.zip: False /content/drive/MyDrive/pet-ai4(3)/dataset.zip
archive.zip: False /content/drive/MyDrive/pet-ai4(3)/archive.zip
IMG_SIZE: 336
BATCH_SIZE: 1
GRAD_ACCUM_STEPS: 8
MAX_EPOCHS_DOG: 70
EARLY_STOP_PATIENCE_DOG: 14
LR_HEAD: 0.0001
LR_BACKBONE: 2e-05
LR_HEAD_HUFEP: 0.0001
LR_BACKBONE_HUFEP: 2e-05
FREEZE_RATIO_HUFEP: 0.0
HUFEP_RANDOM_ONE_CROP_PER_IMAGE: False
HUFEP crop candidates: ['crop_headsafe', 'crop_tight', 'crop_yolo', 'crop_center', 'crop_original']
baseline crops: 

In [ ]:
# =========================================================
# 4. 압축 해제 + human/dog manifest 생성
# =========================================================
def unzip_once(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / ".unzipped_ok"
    if marker.exists():
        print(f"✅ already unzipped: {out_dir}")
        return
    if not zip_path.exists():
        raise FileNotFoundError(f"zip not found: {zip_path}")
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"unzip: {zip_path} -> {out_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
    marker.write_text("ok", encoding="utf-8")

def list_images(root: Path):
    return [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS and p.is_file()]

def split_stratified(df, label_col="label", val_ratio=0.1, test_ratio=0.1, seed=SEED):
    # 먼저 train과 temp 분리, temp를 val/test로 분리
    temp_ratio = val_ratio + test_ratio
    train_df, temp_df = train_test_split(
        df,
        test_size=temp_ratio,
        random_state=seed,
        stratify=df[label_col],
    )
    rel_test = test_ratio / temp_ratio
    val_df, test_df = train_test_split(
        temp_df,
        test_size=rel_test,
        random_state=seed,
        stratify=temp_df[label_col],
    )
    train_df = train_df.copy(); train_df["split"] = "train"
    val_df = val_df.copy(); val_df["split"] = "val"
    test_df = test_df.copy(); test_df["split"] = "test"
    return pd.concat([train_df, val_df, test_df], ignore_index=True)

def build_human_manifest():
    unzip_once(HUMAN_ARCHIVE_ZIP, HUMAN_EXTRACT_DIR)
    rows = []
    for p in list_images(HUMAN_EXTRACT_DIR):
        parts = [x.lower() for x in p.parts]
        label = None
        for c in HUMAN_CLASSES:
            if c in parts:
                label = c
                break
        if label is None:
            parent = p.parent.name.lower()
            if parent in HUMAN_CLASSES:
                label = parent
        if label in HUMAN_CLASSES:
            rows.append({"path": str(p), "label": label, "label_idx": HUMAN_CLASS_TO_IDX[label]})
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("archive.zip에서 사람 표정 이미지/클래스를 찾지 못했습니다.")

    # 사람 표정은 train/val로만 나눔
    train_df, val_df = train_test_split(
        df,
        test_size=0.20,
        random_state=SEED,
        stratify=df["label"],
    )
    train_df = train_df.copy(); train_df["split"] = "train"
    val_df = val_df.copy(); val_df["split"] = "val"
    out = pd.concat([train_df, val_df], ignore_index=True)

    manifest_path = OUTPUT_DIR / "human_archive_manifest.csv"
    out.to_csv(manifest_path, index=False)
    print("✅ human archive classes:", sorted(out["label"].unique().tolist()))
    print(out.groupby(["split", "label"]).size())
    print("✅ saved:", manifest_path)
    return out

def build_dog_manifest():
    unzip_once(DOG_DATASET_ZIP, DOG_EXTRACT_DIR)

    # labels.csv 기반 우선
    csvs = list(DOG_EXTRACT_DIR.rglob("labels.csv")) + list(DOG_EXTRACT_DIR.rglob("label.csv"))
    rows = []

    if csvs:
        csv_path = csvs[0]
        lab = pd.read_csv(csv_path)
        print("✅ dog labels csv:", csv_path, lab.columns.tolist())

        # filename column 추정
        fname_col = None
        for cand in ["filename", "file", "image", "image_id", "path", "filepath"]:
            if cand in lab.columns:
                fname_col = cand; break
        if fname_col is None:
            fname_col = lab.columns[0]

        label_col = None
        for cand in ["label", "emotion", "class", "category"]:
            if cand in lab.columns:
                label_col = cand; break
        if label_col is None and len(lab.columns) >= 2:
            label_col = lab.columns[1]
        if label_col is None:
            raise RuntimeError("labels.csv에서 label column을 찾지 못했습니다.")

        # 파일 인덱스
        image_map = {p.name: p for p in list_images(DOG_EXTRACT_DIR)}
        image_map_lower = {p.name.lower(): p for p in list_images(DOG_EXTRACT_DIR)}

        for _, r in lab.iterrows():
            fname = str(r[fname_col])
            label = str(r[label_col]).strip().lower()
            if label not in DOG_CLASSES:
                continue
            p = image_map.get(Path(fname).name) or image_map_lower.get(Path(fname).name.lower())
            if p is not None:
                rows.append({"path": str(p), "filename": p.name, "label": label, "label_idx": DOG_CLASS_TO_IDX[label]})

    if not rows:
        # 폴더 구조 기반 fallback
        for p in list_images(DOG_EXTRACT_DIR):
            parts = [x.lower() for x in p.parts]
            label = None
            for c in DOG_CLASSES:
                if c in parts:
                    label = c
                    break
            if label in DOG_CLASSES:
                rows.append({"path": str(p), "filename": p.name, "label": label, "label_idx": DOG_CLASS_TO_IDX[label]})

    df = pd.DataFrame(rows).drop_duplicates(subset=["path"])
    if df.empty:
        raise RuntimeError("dataset.zip에서 강아지 이미지/라벨을 찾지 못했습니다.")

    out = split_stratified(df, "label", VAL_RATIO, TEST_RATIO, SEED)
    manifest_path = OUTPUT_DIR / "dog_manifest_split.csv"
    out.to_csv(manifest_path, index=False)

    print("✅ dog manifest:", len(out))
    print(out.groupby(["split", "label"]).size())
    print("✅ saved:", manifest_path)
    return out

human_manifest = build_human_manifest()
dog_manifest = build_dog_manifest()


FileNotFoundError: zip not found: /content/drive/MyDrive/pet-ai4(3)/archive.zip

In [ ]:

# =========================================================
# 5. YOLO detector 로드 — yolo26x 자동 다운로드 + Drive 저장
# =========================================================
from ultralytics import YOLO
from pathlib import Path
import shutil

YOLO26X_DRIVE_PATH = PROJECT_BASE / "yolo26x.pt"
YOLO26N_DRIVE_PATH = PROJECT_BASE / "yolo26n.pt"
YOLOV8X_DRIVE_PATH = PROJECT_BASE / "yolov8x.pt"

def prepare_detector_yolo26x():
    print("\n==============================")
    print("YOLO detector 준비")
    print("==============================")
    print("PROJECT_BASE:", PROJECT_BASE)
    print("yolo26x.pt:", YOLO26X_DRIVE_PATH.exists(), YOLO26X_DRIVE_PATH)
    print("yolo26n.pt:", YOLO26N_DRIVE_PATH.exists(), YOLO26N_DRIVE_PATH)
    print("yolov8x.pt:", YOLOV8X_DRIVE_PATH.exists(), YOLOV8X_DRIVE_PATH)

    if YOLO26X_DRIVE_PATH.exists():
        print("✅ Drive yolo26x.pt 사용:", YOLO26X_DRIVE_PATH)
        return YOLO(str(YOLO26X_DRIVE_PATH)), YOLO26X_DRIVE_PATH

    print("⚠️ Drive에 yolo26x.pt 없음 → Ultralytics 자동 다운로드 시도")
    try:
        detector = YOLO("yolo26x.pt")
        candidates = [
            Path("yolo26x.pt"),
            Path("/content/yolo26x.pt"),
            Path.home() / ".cache" / "ultralytics" / "yolo26x.pt",
        ]
        found = next((p for p in candidates if p.exists()), None)
        if found is not None:
            shutil.copy2(found, YOLO26X_DRIVE_PATH)
            print("✅ yolo26x.pt Drive 저장 완료:", YOLO26X_DRIVE_PATH)
            return YOLO(str(YOLO26X_DRIVE_PATH)), YOLO26X_DRIVE_PATH
        print("✅ yolo26x.pt 모델 로드 성공. 파일 위치는 못 찾았지만 이번 런타임에서 사용합니다.")
        return detector, Path("yolo26x.pt")
    except Exception as e:
        print("⚠️ yolo26x.pt 자동 다운로드/로드 실패:", repr(e))

    if YOLO26N_DRIVE_PATH.exists():
        print("⚠️ yolo26x 실패 → yolo26n.pt 사용:", YOLO26N_DRIVE_PATH)
        return YOLO(str(YOLO26N_DRIVE_PATH)), YOLO26N_DRIVE_PATH

    if YOLOV8X_DRIVE_PATH.exists():
        print("⚠️ yolo26x 실패 → yolov8x.pt 사용:", YOLOV8X_DRIVE_PATH)
        return YOLO(str(YOLOV8X_DRIVE_PATH)), YOLOV8X_DRIVE_PATH

    print("⚠️ yolo26x/yolo26n 모두 실패 → fallback yolov8x.pt 자동 다운로드")
    detector = YOLO("yolov8x.pt")
    if Path("yolov8x.pt").exists():
        shutil.copy2("yolov8x.pt", YOLOV8X_DRIVE_PATH)
        print("✅ yolov8x.pt Drive 저장:", YOLOV8X_DRIVE_PATH)
    return detector, YOLOV8X_DRIVE_PATH if YOLOV8X_DRIVE_PATH.exists() else Path("yolov8x.pt")


detector, DETECTOR_MODEL_PATH = prepare_detector_yolo26x()
print("\n✅ 최종 detector path:", DETECTOR_MODEL_PATH)
print("==============================")


In [ ]:
# =========================================================
# 6. YOLO crop 함수
# =========================================================
import cv2

def safe_open_rgb(path):
    img = Image.open(path).convert("RGB")
    return img

def clip_box(x1, y1, x2, y2, w, h):
    x1 = max(0, min(int(round(x1)), w - 1))
    y1 = max(0, min(int(round(y1)), h - 1))
    x2 = max(x1 + 1, min(int(round(x2)), w))
    y2 = max(y1 + 1, min(int(round(y2)), h))
    return x1, y1, x2, y2

def expand_box(box, w, h, pad=0.2):
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    return clip_box(x1 - bw * pad, y1 - bh * pad, x2 + bw * pad, y2 + bh * pad, w, h)

def head_box_from_dog_box(box, w, h, top_ratio=0.60, pad=0.10):
    x1, y1, x2, y2 = box
    hh = (y2 - y1) * top_ratio
    hb = (x1, y1, x2, y1 + hh)
    return expand_box(hb, w, h, pad=pad)

def center_square_box(w, h, scale=0.85):
    side = min(w, h) * scale
    cx, cy = w / 2, h / 2
    return clip_box(cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2, w, h)

def crop_save(img: Image.Image, box, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    crop = img.crop(box)
    crop.save(out_path, quality=95)
    return str(out_path)

def detect_best_dog_box(image_path: str):
    img = safe_open_rgb(image_path)
    w, h = img.size
    try:
        res = detector.predict(
            source=image_path,
            conf=DETECT_CONF,
            iou=DETECT_IOU,
            imgsz=DETECT_IMGSZ,
            verbose=False,
            classes=[16],
        )[0]
        boxes = []
        if res.boxes is not None and len(res.boxes) > 0:
            for b in res.boxes:
                xyxy = b.xyxy.detach().cpu().numpy()[0]
                conf = float(b.conf.detach().cpu().numpy()[0])
                x1, y1, x2, y2 = xyxy
                area = max(0, x2 - x1) * max(0, y2 - y1)
                if area / (w * h) >= MIN_BOX_AREA_RATIO:
                    boxes.append((conf, area, (x1, y1, x2, y2)))
        if boxes:
            boxes.sort(key=lambda x: (x[0], x[1]), reverse=True)
            conf, area, box = boxes[0]
            return clip_box(*box, w, h), conf
    except Exception as e:
        print("detect failed:", image_path, e)
    return None, 0.0

def make_crops_for_row(row):
    image_path = row["path"]
    split = row["split"]
    label = row["label"]
    stem = hashlib.md5(image_path.encode("utf-8")).hexdigest()[:16]

    img = safe_open_rgb(image_path)
    w, h = img.size
    dog_box, dog_conf = detect_best_dog_box(image_path)

    boxes = {}
    boxes["crop_original"] = (0, 0, w, h)
    boxes["crop_center"] = center_square_box(w, h, 0.85)

    if dog_box is None:
        boxes["crop_yolo"] = boxes["crop_center"]
        boxes["crop_tight"] = boxes["crop_center"]
        boxes["crop_headsafe"] = boxes["crop_center"]
        detected = False
    else:
        boxes["crop_yolo"] = expand_box(dog_box, w, h, CROP_PADDING)
        boxes["crop_tight"] = expand_box(dog_box, w, h, TIGHT_PADDING)
        boxes["crop_headsafe"] = head_box_from_dog_box(dog_box, w, h, HEAD_TOP_RATIO, pad=0.12)
        detected = True

    out = {
        "detected": detected,
        "det_conf": dog_conf,
    }
    for cname, box in boxes.items():
        out_path = CROP_CACHE_DIR / split / cname / label / f"{stem}.jpg"
        out[cname] = crop_save(img, box, out_path)
    return out


In [ ]:

# =========================================================
# 7. crop cache 생성/재사용 + 파일 유실 자동 복구
# =========================================================
crop_manifest_path = OUTPUT_DIR / "dog_crop_manifest.csv"
# 518 새 실험도 crop cache는 이미지 파일 자체라 기존 224 manifest가 있으면 재사용 가능
old_crop_manifest_path = OLD_OUTPUT_DIR / "dog_crop_manifest.csv"

def crop_file_exists_count(df, cols):
    cnt = 0
    for c in cols:
        if c in df.columns:
            cnt += int(df[c].apply(lambda x: isinstance(x, str) and Path(x).exists()).sum())
    return cnt

def build_crop_manifest():
    rows = []
    for _, r in tqdm(dog_manifest.iterrows(), total=len(dog_manifest), desc="YOLO crop cache"):
        base = r.to_dict()
        crops = make_crops_for_row(base)
        base.update(crops)
        rows.append(base)
    return pd.DataFrame(rows)

need_rebuild = True
if crop_manifest_path.exists():
    dog_crop_manifest = pd.read_csv(crop_manifest_path)
    train_df_tmp = dog_crop_manifest[dog_crop_manifest["split"] == "train"].copy()
    existing = crop_file_exists_count(train_df_tmp, HUFEP_VITH14_DOG_TRAIN_CROP_COLS)
    print("✅ crop manifest loaded:", crop_manifest_path, len(dog_crop_manifest), "existing train crop files:", existing)
    need_rebuild = existing == 0
elif old_crop_manifest_path.exists():
    dog_crop_manifest = pd.read_csv(old_crop_manifest_path)
    train_df_tmp = dog_crop_manifest[dog_crop_manifest["split"] == "train"].copy()
    existing = crop_file_exists_count(train_df_tmp, HUFEP_VITH14_DOG_TRAIN_CROP_COLS)
    print("✅ old crop manifest loaded:", old_crop_manifest_path, len(dog_crop_manifest), "existing train crop files:", existing)
    need_rebuild = existing == 0
    if not need_rebuild:
        dog_crop_manifest.to_csv(crop_manifest_path, index=False)
        print("✅ copied old manifest to new output:", crop_manifest_path)
else:
    need_rebuild = True

if need_rebuild:
    print("⚠️ crop manifest가 없거나 crop 파일이 사라졌습니다. crop cache를 다시 생성합니다.")
    dog_crop_manifest = build_crop_manifest()
    dog_crop_manifest.to_csv(crop_manifest_path, index=False)
    print("✅ crop manifest saved:", crop_manifest_path)

print(dog_crop_manifest.groupby(["split", "label"]).size())
print("detected ratio:", dog_crop_manifest["detected"].mean())
print("avg det_conf:", dog_crop_manifest["det_conf"].mean())
print("train existing crop files:", crop_file_exists_count(dog_crop_manifest[dog_crop_manifest["split"] == "train"], HUFEP_VITH14_DOG_TRAIN_CROP_COLS))


In [ ]:

# =========================================================
# 8. Dataset / Transform
# =========================================================
def get_timm_transforms(model_name=VITH14_MODEL_NAME, train=True):
    cfg = resolve_model_data_config(timm.create_model(model_name, pretrained=False))
    if train:
        return create_transform(
            input_size=(3, IMG_SIZE, IMG_SIZE),
            is_training=True,
            mean=cfg.get("mean", (0.5, 0.5, 0.5)),
            std=cfg.get("std", (0.5, 0.5, 0.5)),
            interpolation=cfg.get("interpolation", "bicubic"),
            auto_augment="rand-m9-mstd0.5-inc1",
            re_prob=0.10,
        )
    return create_transform(
        input_size=(3, IMG_SIZE, IMG_SIZE),
        is_training=False,
        mean=cfg.get("mean", (0.5, 0.5, 0.5)),
        std=cfg.get("std", (0.5, 0.5, 0.5)),
        interpolation=cfg.get("interpolation", "bicubic"),
    )

train_tf = get_timm_transforms(train=True)
eval_tf = get_timm_transforms(train=False)

class SimpleImageDataset(Dataset):
    def __init__(self, df, classes, transform):
        self.df = df.reset_index(drop=True).copy()
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = safe_open_rgb(r["path"])
        x = self.transform(img)
        y = int(self.class_to_idx[str(r["label"])])
        return x, y

class DogCropDataset(Dataset):
    def __init__(self, df, crop_cols, transform, expand=True, detected_only=False, min_conf=0.0):
        src = df.reset_index(drop=True).copy()
        if detected_only:
            src = src[(src["detected"] == True) & (src["det_conf"] >= min_conf)].copy()
            if src.empty:
                raise RuntimeError("detected_only 조건을 만족하는 dog crop 데이터가 없습니다.")

        samples = []
        if expand:
            for _, r in src.iterrows():
                for c in crop_cols:
                    if c in r and isinstance(r[c], str) and Path(r[c]).exists():
                        samples.append({"path": r[c], "label": r["label"], "label_idx": int(r["label_idx"]), "crop_col": c})
        else:
            for _, r in src.iterrows():
                for c in crop_cols:
                    if c in r and isinstance(r[c], str) and Path(r[c]).exists():
                        samples.append({"path": r[c], "label": r["label"], "label_idx": int(r["label_idx"]), "crop_col": c})
                        break

        self.df = pd.DataFrame(samples)
        self.transform = transform
        print(f"✅ DogCropDataset: base={len(src)} -> samples={len(self.df)} cols={crop_cols}")
        if len(self.df) == 0:
            raise RuntimeError("DogCropDataset samples=0입니다. 7번 crop cache 재생성을 먼저 실행하세요.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = safe_open_rgb(r["path"])
        return self.transform(img), int(r["label_idx"])

class RandomMultiCropDogDataset(Dataset):
    """
    5개 crop 후보는 유지하되 이미지 1장당 매번 crop 1개만 랜덤 선택.
    3200장 train이면 batch 1 기준 약 3200 step.
    """
    def __init__(self, df, crop_cols, transform, detected_only=False, min_conf=0.0):
        src = df.reset_index(drop=True).copy()
        if detected_only:
            src = src[(src["detected"] == True) & (src["det_conf"] >= min_conf)].copy()
            if src.empty:
                raise RuntimeError("detected_only 조건을 만족하는 dog crop 데이터가 없습니다.")

        samples = []
        for _, r in src.iterrows():
            valid_cols = []
            for c in crop_cols:
                if c in r and isinstance(r[c], str) and Path(r[c]).exists():
                    valid_cols.append(c)
            if valid_cols:
                samples.append({
                    "label": r["label"],
                    "label_idx": int(r["label_idx"]),
                    "crop_cols": valid_cols,
                    "row": r.to_dict(),
                })

        self.df = pd.DataFrame(samples)
        self.transform = transform
        print(f"✅ RandomMultiCropDogDataset: base={len(src)} -> samples={len(self.df)} random_cols={crop_cols}")
        if len(self.df) == 0:
            raise RuntimeError("RandomMultiCropDogDataset samples=0입니다. 7번 crop cache 재생성을 먼저 실행하세요.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        chosen_col = random.choice(r["crop_cols"])
        img_path = r["row"][chosen_col]
        img = safe_open_rgb(img_path)
        return self.transform(img), int(r["label_idx"])


class MultiCropSimultaneousDogDataset(Dataset):
    """
    이미지 1장당 5개 crop을 동시에 반환합니다.

    반환 shape:
        x: [num_crops, C, H, W]
        y: int

    그래서 train_df가 3200장이면 dataset 길이도 3200입니다.
    하지만 한 step에서 모델은 crop 5개를 모두 보고, logits를 가중 평균해서 loss를 계산합니다.
    """
    def __init__(self, df, crop_cols, transform, detected_only=False, min_conf=0.0, require_all=False):
        src = df.reset_index(drop=True).copy()
        if detected_only:
            src = src[(src["detected"] == True) & (src["det_conf"] >= min_conf)].copy()
            if src.empty:
                raise RuntimeError("detected_only 조건을 만족하는 dog crop 데이터가 없습니다.")

        samples = []
        for _, r in src.iterrows():
            paths = []
            valid = []
            for c in crop_cols:
                p = r[c] if c in r else None
                if isinstance(p, str) and Path(p).exists():
                    paths.append(p)
                    valid.append(c)
                else:
                    paths.append(None)

            if not valid:
                continue
            if require_all and len(valid) != len(crop_cols):
                continue

            # 일부 crop 파일이 없으면 첫 valid crop으로 대체해서 항상 5개 shape를 유지합니다.
            fallback_path = r[valid[0]]
            fixed_paths = [p if p is not None else fallback_path for p in paths]
            samples.append({
                "label": r["label"],
                "label_idx": int(r["label_idx"]),
                "paths": fixed_paths,
                "valid_crop_cols": valid,
            })

        self.df = pd.DataFrame(samples)
        self.crop_cols = list(crop_cols)
        self.transform = transform
        print(f"✅ MultiCropSimultaneousDogDataset: base={len(src)} -> images={len(self.df)} crops_per_image={len(self.crop_cols)} cols={crop_cols}")
        if len(self.df) == 0:
            raise RuntimeError("MultiCropSimultaneousDogDataset samples=0입니다. 7번 crop cache 재생성을 먼저 실행하세요.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        xs = []
        for p in r["paths"]:
            img = safe_open_rgb(p)
            xs.append(self.transform(img))
        x = torch.stack(xs, dim=0)  # [K, C, H, W]
        y = int(r["label_idx"])
        return x, y

def make_loader(ds, train=True, batch_size=BATCH_SIZE, sampler=None):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(train and sampler is None),
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=False,
    )


In [ ]:
# =========================================================
# 9. 모델 / 학습 유틸
# =========================================================
def resize_pos_embed_if_needed(src_state, model):
    """224 checkpoint의 pos_embed를 현재 IMG_SIZE 모델 크기에 맞게 보간해서 로드합니다."""
    if "pos_embed" not in src_state or not hasattr(model, "pos_embed"):
        return src_state
    pe = src_state["pos_embed"]
    target = model.pos_embed
    if pe.shape == target.shape:
        return src_state
    try:
        cls_tok = pe[:, :1]
        patch_tok = pe[:, 1:]
        old_n = patch_tok.shape[1]
        new_n = target.shape[1] - 1
        old_hw = int(math.sqrt(old_n))
        new_hw = int(math.sqrt(new_n))
        if old_hw * old_hw != old_n or new_hw * new_hw != new_n:
            print("⚠️ pos_embed grid가 square가 아니라서 pos_embed는 제외합니다.")
            src_state = {k: v for k, v in src_state.items() if k != "pos_embed"}
            return src_state
        patch_tok = patch_tok.reshape(1, old_hw, old_hw, -1).permute(0, 3, 1, 2)
        patch_tok = torch.nn.functional.interpolate(patch_tok, size=(new_hw, new_hw), mode="bicubic", align_corners=False)
        patch_tok = patch_tok.permute(0, 2, 3, 1).reshape(1, new_hw * new_hw, -1)
        src_state = dict(src_state)
        src_state["pos_embed"] = torch.cat([cls_tok, patch_tok], dim=1)
        print(f"✅ pos_embed resized: {tuple(pe.shape)} -> {tuple(src_state['pos_embed'].shape)}")
        return src_state
    except Exception as e:
        print("⚠️ pos_embed resize 실패, pos_embed 제외:", repr(e))
        src_state = {k: v for k, v in src_state.items() if k != "pos_embed"}
        return src_state

def load_flexible_state_dict(model, state, exclude_head=False):
    if isinstance(state, dict) and "model" in state:
        state = state["model"]
    if exclude_head:
        state = {k: v for k, v in state.items() if not (k.startswith("head.") or k.startswith("fc.") or k.startswith("classifier."))}
    state = resize_pos_embed_if_needed(state, model)
    model_state = model.state_dict()
    filtered = {}
    skipped = []
    for k, v in state.items():
        if k in model_state and tuple(model_state[k].shape) == tuple(v.shape):
            filtered[k] = v
        else:
            skipped.append(k)
    msg = model.load_state_dict(filtered, strict=False)
    if skipped:
        print("⚠️ skipped mismatched keys:", skipped[:10], "... total", len(skipped))
    return msg

def create_vith14(num_classes):
    model = timm.create_model(VITH14_MODEL_NAME, pretrained=True, num_classes=num_classes, img_size=IMG_SIZE)
    if hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(True)
        print("✅ grad checkpointing enabled")
    return model

def set_trainable_head_last(model, last_n=1):
    for p in model.parameters():
        p.requires_grad = False
    if hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True
    if hasattr(model, "norm"):
        for p in model.norm.parameters():
            p.requires_grad = True
    if hasattr(model, "blocks"):
        for blk in model.blocks[-last_n:]:
            for p in blk.parameters():
                p.requires_grad = True

def set_trainable_full(model):
    for p in model.parameters():
        p.requires_grad = True

def count_trainable(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

def get_optimizer(model):
    head_params, backbone_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "head" in n:
            head_params.append(p)
        else:
            backbone_params.append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    if head_params:
        groups.append({"params": head_params, "lr": LR_HEAD})
    print(f"optimizer groups: backbone_tensors={len(backbone_params)} lr={LR_BACKBONE}, head_tensors={len(head_params)} lr={LR_HEAD}")
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def compute_class_weights(labels, num_classes, power=0.5, clip_max=4.0):
    counts = np.bincount(np.array(labels, dtype=int), minlength=num_classes)
    counts = np.maximum(counts, 1)
    inv = (counts.sum() / (num_classes * counts)) ** power
    inv = np.clip(inv, 0.0, clip_max)
    inv = inv / inv.mean()
    return torch.tensor(inv, dtype=torch.float32), counts

def make_weighted_sampler(labels, num_classes, power=1.0):
    counts = np.bincount(np.array(labels, dtype=int), minlength=num_classes)
    counts = np.maximum(counts, 1)
    class_w = (counts.sum() / (num_classes * counts)) ** power
    sample_w = np.array([class_w[int(y)] for y in labels], dtype=np.float64)
    return WeightedRandomSampler(weights=sample_w, num_samples=len(labels), replacement=True)


def _get_logits(outputs):
    if isinstance(outputs, torch.Tensor):
        return outputs
    if hasattr(outputs, "logits"):
        return outputs.logits
    if isinstance(outputs, dict) and "logits" in outputs:
        return outputs["logits"]
    return outputs

def forward_multicrop_logits(model, x, crop_weights=None):
    """
    4D input: [B, C, H, W] -> 일반 단일 crop forward
    5D input: [B, K, C, H, W] -> K개 crop을 모두 forward한 뒤 weighted logits 평균

    학습에서 5개 crop을 전부 '동시에' 반영하되, dataset 길이는 이미지 수 그대로 유지합니다.
    """
    if x.dim() == 5:
        b, k, c, h, w = x.shape
        flat = x.reshape(b * k, c, h, w)
        logits = _get_logits(model(flat)).reshape(b, k, -1)
        if crop_weights is None:
            weights = globals().get("MULTICROP_WEIGHT_LIST", None)
        else:
            weights = crop_weights
        if weights is None or len(weights) != k:
            ww = torch.ones(k, device=logits.device, dtype=logits.dtype) / float(k)
        else:
            ww = torch.tensor(weights, device=logits.device, dtype=logits.dtype)
            ww = ww / ww.sum().clamp_min(1e-8)
        return (logits * ww.view(1, k, 1)).sum(dim=1)
    return _get_logits(model(x))

@torch.no_grad()
def evaluate_model(model, loader, num_classes):
    model.eval()
    ys, ps = [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP and torch.cuda.is_available()):
            logits = forward_multicrop_logits(model, x)
            loss = criterion(logits, y)
        pred = logits.argmax(dim=1)
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(pred.detach().cpu().numpy().tolist())
        total_loss += float(loss.detach().cpu()) * len(y)
    return {
        "val_loss": total_loss / max(1, len(ys)),
        "val_accuracy": accuracy_score(ys, ps),
        "val_balanced_accuracy": balanced_accuracy_score(ys, ps),
        "val_macro_f1": f1_score(ys, ps, average="macro"),
        "y_true": ys,
        "y_pred": ps,
    }

def save_reports(out_dir, classes, y_true, y_pred, prefix="val"):
    out_dir.mkdir(parents=True, exist_ok=True)
    rep = classification_report(y_true, y_pred, target_names=classes, output_dict=True, zero_division=0)
    with open(out_dir / f"{prefix}_report.json", "w", encoding="utf-8") as f:
        json.dump(rep, f, ensure_ascii=False, indent=2)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title(f"{prefix} confusion matrix")
    plt.xticks(range(len(classes)), classes, rotation=45)
    plt.yticks(range(len(classes)), classes)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(out_dir / f"{prefix}_confusion_matrix.png", dpi=160)
    plt.close()

def train_loop(
    model,
    train_loader,
    val_loader,
    out_dir: Path,
    classes,
    max_epochs,
    patience,
    stage_switch_epoch=5,
    full_after_switch=True,
    class_weight_tensor=None,
    resume=True,
    run_name="model",
):
    out_dir.mkdir(parents=True, exist_ok=True)
    best_path = out_dir / "best_model.pt"
    last_path = out_dir / "last_checkpoint.pt"

    model = model.to(DEVICE)
    set_trainable_head_last(model, last_n=1)
    tr, total = count_trainable(model)
    print(f"{run_name} trainable start: {tr:,}/{total:,} ({tr/total*100:.2f}%)")

    optimizer = get_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

    start_epoch = 1
    best_f1 = -1.0
    no_improve = 0

    if resume and last_path.exists():
        ckpt = torch.load(last_path, map_location="cpu")
        model.load_state_dict(ckpt["model"], strict=False)
        best_f1 = float(ckpt.get("best_f1", -1.0))
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        print(f"✅ resume: {last_path} epoch {start_epoch} best_f1 {best_f1:.5f}")

    if class_weight_tensor is not None:
        class_weight_tensor = class_weight_tensor.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor, label_smoothing=LABEL_SMOOTHING)

    for epoch in range(start_epoch, max_epochs + 1):
        if epoch == stage_switch_epoch + 1:
            print("\n========== Switch to V3-style FINE-TUNING ==========")
            if full_after_switch:
                set_trainable_full(model)
                print("🔓 full fine-tuning")
            else:
                set_trainable_head_last(model, last_n=16)
                print("🔓 last16 fine-tuning")
            optimizer = get_optimizer(model)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

        model.train()
        loss_sum, correct, n = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f"{run_name} epoch {epoch}/{max_epochs}")
        for step, (x, y) in enumerate(pbar, start=1):
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_AMP and torch.cuda.is_available()):
                logits = forward_multicrop_logits(model, x)
                loss = criterion(logits, y) / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            real_loss = float(loss.detach().cpu()) * GRAD_ACCUM_STEPS
            pred = logits.argmax(dim=1)
            bs = len(y)
            loss_sum += real_loss * bs
            correct += int((pred == y).sum().detach().cpu())
            n += bs
            pbar.set_postfix(loss=loss_sum / max(1, n), acc=correct / max(1, n))

        scheduler.step()

        val = evaluate_model(model, val_loader, len(classes))
        log = {
            "epoch": epoch,
            "train_loss": loss_sum / max(1, n),
            "train_acc": correct / max(1, n),
            "lr_head": optimizer.param_groups[-1]["lr"],
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "val_accuracy": val["val_accuracy"],
            "val_balanced_accuracy": float(val["val_balanced_accuracy"]),
            "val_macro_f1": val["val_macro_f1"],
        }
        print("\n", run_name, log)

        is_best = val["val_macro_f1"] > best_f1
        if is_best:
            best_f1 = val["val_macro_f1"]
            no_improve = 0
            torch.save({
                "model": model.state_dict(),
                "model_name": VITH14_MODEL_NAME,
                "classes": classes,
                "epoch": epoch,
                "best_f1": best_f1,
                "config": {
                    "IMG_SIZE": IMG_SIZE,
                    "LR_HEAD": LR_HEAD,
                    "LR_BACKBONE": LR_BACKBONE,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "LABEL_SMOOTHING": LABEL_SMOOTHING,
                }
            }, best_path)
            save_reports(out_dir, classes, val["y_true"], val["y_pred"], prefix="val")
            print(f"🔥 BEST saved: {best_path} macro_f1: {best_f1:.5f}")
        else:
            no_improve += 1
            print(f"⏳ no improve: {no_improve}/{patience} best_macro_f1={best_f1:.5f}")

        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "best_f1": best_f1,
        }, last_path)

        if no_improve >= patience:
            print(f"🛑 early stopping: {run_name}")
            break

    if best_path.exists():
        ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"], strict=False)
        print("✅ loaded best weights:", best_path)
    return model, best_f1


In [ ]:

# =========================================================
# 10. 예측 / 앙상블 유틸
# - validation/test에서는 crop별 probability를 구한 뒤 crop weight로 평균합니다.
# =========================================================
@torch.no_grad()
def predict_probs_for_df(model, df, crop_cols, classes, batch_size=BATCH_SIZE, crop_weights=None):
    model.eval().to(DEVICE)
    probs_acc = None
    weight_acc = 0.0
    y_true = None

    if crop_weights is None:
        crop_weights = globals().get("CROP_WEIGHTS", {})

    for crop_col in crop_cols:
        if crop_col not in df.columns:
            continue
        tmp = df.copy()
        tmp["path"] = tmp[crop_col]
        tmp = tmp[tmp["path"].apply(lambda x: isinstance(x, str) and Path(x).exists())].copy()
        if tmp.empty:
            continue

        ds = DogCropDataset(tmp, [crop_col], eval_tf, expand=False)
        loader = make_loader(ds, train=False, batch_size=batch_size)

        all_probs, all_y = [], []
        for x, y in tqdm(loader, desc=f"predict {crop_col}"):
            x = x.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP and torch.cuda.is_available()):
                logits = forward_multicrop_logits(model, x)
                prob = torch.softmax(logits, dim=1)
            all_probs.append(prob.detach().cpu().numpy())
            all_y.extend(y.numpy().tolist())

        p = np.concatenate(all_probs, axis=0)
        w = float(crop_weights.get(crop_col, 1.0)) if isinstance(crop_weights, dict) else 1.0
        if probs_acc is None:
            probs_acc = p * w
            y_true = np.array(all_y)
        else:
            probs_acc += p * w
        weight_acc += w

    if probs_acc is None:
        raise RuntimeError(f"사용 가능한 crop이 없습니다: {crop_cols}")
    return probs_acc / max(1e-8, weight_acc), y_true

def metrics_from_probs(probs, y_true, classes):
    pred = probs.argmax(axis=1)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "pred": pred,
    }

def load_vith14_checkpoint(path, num_classes):
    model = timm.create_model(VITH14_MODEL_NAME, pretrained=False, num_classes=num_classes, img_size=IMG_SIZE)
    if hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(True)
    ckpt = torch.load(path, map_location="cpu")
    load_flexible_state_dict(model, ckpt, exclude_head=False)
    return model.to(DEVICE)


In [ ]:

# =========================================================
# 10-X. HuFEP 전용 stable 학습 함수
# - 기존 train_loop를 건드리지 않음
# - HuFEP 336 + 5crop simultaneous 설정 사용
# =========================================================

def _get_logits(outputs):
    if isinstance(outputs, torch.Tensor):
        return outputs
    if hasattr(outputs, "logits"):
        return outputs.logits
    if isinstance(outputs, dict) and "logits" in outputs:
        return outputs["logits"]
    return outputs

def apply_hufep_freeze(model, freeze_ratio=0.0):
    if not hasattr(model, "blocks"):
        print("⚠️ blocks not found, skip HuFEP freeze")
        return model
    total_blocks = len(model.blocks)
    freeze_until = int(total_blocks * freeze_ratio)
    for i, block in enumerate(model.blocks):
        trainable = i >= freeze_until
        for p in block.parameters():
            p.requires_grad = trainable
    for name, p in model.named_parameters():
        if "head" in name or "fc" in name or "classifier" in name:
            p.requires_grad = True
    print(f"✅ HuFEP freeze applied: {freeze_until}/{total_blocks} blocks frozen")
    return model

def unfreeze_all_hufep(model):
    for p in model.parameters():
        p.requires_grad = True
    print("🔥 HuFEP all layers unfrozen")
    return model

def build_hufep_optimizer(model, lr_backbone=None, lr_head=None):
    if lr_backbone is None:
        lr_backbone = LR_BACKBONE_HUFEP
    if lr_head is None:
        lr_head = LR_HEAD_HUFEP
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "head" in name or "fc" in name or "classifier" in name:
            head_params.append(p)
        else:
            backbone_params.append(p)
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    print(f"✅ HuFEP optimizer: backbone_tensors={len(backbone_params)} lr={lr_backbone}, head_tensors={len(head_params)} lr={lr_head}")
    return optimizer

def build_hufep_scheduler(optimizer, total_epochs):
    if not USE_COSINE_HUFEP:
        return None
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, int(total_epochs)), eta_min=MIN_LR_HUFEP)

@torch.no_grad()
def evaluate_hufep_stable(model, loader, classes):
    model.eval()
    preds, gts = [], []
    total_loss, total_n = 0.0, 0
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP and torch.cuda.is_available()):
            logits = forward_multicrop_logits(model, x)
            loss = criterion(logits, y)
        pred = logits.argmax(dim=1)
        total_loss += float(loss.detach().cpu()) * len(y)
        total_n += len(y)
        preds.extend(pred.detach().cpu().numpy().tolist())
        gts.extend(y.detach().cpu().numpy().tolist())
    return {
        "val_loss": total_loss / max(1, total_n),
        "val_accuracy": accuracy_score(gts, preds),
        "val_balanced_accuracy": balanced_accuracy_score(gts, preds),
        "val_macro_f1": f1_score(gts, preds, average="macro"),
        "y_true": gts,
        "y_pred": preds,
    }

def save_hufep_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_macro_f1, classes, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": int(epoch),
        "best_f1": float(best_macro_f1),
        "best_macro_f1": float(best_macro_f1),
        "classes": list(classes),
        "model_name": VITH14_MODEL_NAME,
        "config": {
            "IMG_SIZE": IMG_SIZE,
            "LR_HEAD_HUFEP": LR_HEAD_HUFEP,
            "LR_BACKBONE_HUFEP": LR_BACKBONE_HUFEP,
            "FREEZE_RATIO_HUFEP": FREEZE_RATIO_HUFEP,
            "HUFEP_RANDOM_ONE_CROP_PER_IMAGE": HUFEP_RANDOM_ONE_CROP_PER_IMAGE,
            "HUFEP_VITH14_DOG_TRAIN_CROP_COLS": HUFEP_VITH14_DOG_TRAIN_CROP_COLS,
        },
        "extra": extra or {},
    }, path)

def train_hufep_stable_loop(model, train_loader, val_loader, save_dir, classes, max_epochs=None, patience=None, resume=True, run_name="HuFEP-Archive-ViTH14-DogFace-STABLE-336-5CROP-SIMUL"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    if max_epochs is None:
        max_epochs = MAX_EPOCHS_DOG_HUFEP
    if patience is None:
        patience = EARLY_STOP_PATIENCE_DOG_HUFEP

    best_path = save_dir / "best_model.pt"
    last_path = save_dir / "last_checkpoint.pt"
    history_path = save_dir / "history.json"

    model = model.to(DEVICE)
    model = apply_hufep_freeze(model, FREEZE_RATIO_HUFEP)
    optimizer = build_hufep_optimizer(model, LR_BACKBONE_HUFEP, LR_HEAD_HUFEP)
    scheduler = build_hufep_scheduler(optimizer, max_epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

    start_epoch = 0
    best_macro_f1 = -1.0
    no_improve = 0
    history = []
    already_unfrozen = FREEZE_RATIO_HUFEP == 0.0

    if resume and last_path.exists():
        ckpt = torch.load(last_path, map_location="cpu")
        load_flexible_state_dict(model, ckpt, exclude_head=False)
        start_epoch = int(ckpt.get("epoch", 0))
        best_macro_f1 = float(ckpt.get("best_macro_f1", ckpt.get("best_f1", -1.0)))
        print(f"✅ resume stable HuFEP: {last_path} epoch={start_epoch} best_f1={best_macro_f1:.5f}")

    for epoch in range(start_epoch + 1, max_epochs + 1):
        if (not already_unfrozen) and epoch == UNFREEZE_EPOCH_HUFEP:
            print("\n========== HuFEP STABLE: final unfreeze ==========")
            model = unfreeze_all_hufep(model)
            optimizer = build_hufep_optimizer(model, LR_FINAL_UNFREEZE_HUFEP, LR_FINAL_UNFREEZE_HUFEP)
            scheduler = build_hufep_scheduler(optimizer, max_epochs - epoch + 1)
            already_unfrozen = True

        model.train()
        loss_sum, correct, n = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"{run_name} epoch {epoch}/{max_epochs}")
        for step, (x, y) in enumerate(pbar, start=1):
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP and torch.cuda.is_available()):
                logits = forward_multicrop_logits(model, x)
                loss = criterion(logits, y)
                loss_backward = loss / max(1, GRAD_ACCUM_STEPS)
            scaler.scale(loss_backward).backward()
            if step % max(1, GRAD_ACCUM_STEPS) == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                if GRAD_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            pred = logits.argmax(dim=1)
            loss_sum += float(loss.detach().cpu()) * len(y)
            correct += int((pred == y).sum().detach().cpu())
            n += len(y)
            pbar.set_postfix(loss=loss_sum / max(1, n), acc=correct / max(1, n))

        if scheduler is not None:
            scheduler.step()

        val = evaluate_hufep_stable(model, val_loader, classes)
        log = {
            "epoch": epoch,
            "train_loss": loss_sum / max(1, n),
            "train_acc": correct / max(1, n),
            "lr_head": optimizer.param_groups[-1]["lr"],
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "val_accuracy": val["val_accuracy"],
            "val_balanced_accuracy": float(val["val_balanced_accuracy"]),
            "val_macro_f1": val["val_macro_f1"],
        }
        print("\n", run_name, log)
        history.append(log)

        save_hufep_checkpoint(last_path, model, optimizer, scheduler, scaler, epoch, max(best_macro_f1, val["val_macro_f1"]), classes, extra={"type": "last"})

        if val["val_macro_f1"] > best_macro_f1:
            best_macro_f1 = val["val_macro_f1"]
            no_improve = 0
            save_hufep_checkpoint(best_path, model, optimizer, scheduler, scaler, epoch, best_macro_f1, classes, extra={"type": "best"})
            save_reports(save_dir, classes, val["y_true"], val["y_pred"], prefix="val")
            print(f"🔥 BEST STABLE HuFEP saved: {best_path} macro_f1: {best_macro_f1:.5f}")
        else:
            no_improve += 1
            print(f"⏳ no improve: {no_improve}/{patience} best_macro_f1={best_macro_f1:.5f}")
            if no_improve >= patience:
                print("⛔ Early stopping stable HuFEP")
                break

        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)

    if best_path.exists():
        ckpt = torch.load(best_path, map_location="cpu")
        load_flexible_state_dict(model, ckpt, exclude_head=False)
        print("✅ loaded best stable HuFEP:", best_path)
    return model, best_macro_f1


In [ ]:

# =========================================================
# 11-0. Human HuFEP pretrain 준비
# - 기존 224 human HuFEP checkpoint가 있으면 재사용/복사
# - 없으면 archive.zip 사람 표정 데이터로 ViT-H/14 human pretrain 실행
# =========================================================
from pathlib import Path
import shutil, gc

HUMAN_PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
human_best = HUMAN_PRETRAIN_DIR / "best_model.pt"

candidate_old_human = [
    OLD_HUMAN_PRETRAIN_DIR / "best_model.pt",
    OLD_OUTPUT_DIR / "00_archive_HuFEP_ViTH14_human_pretrain" / "best_model.pt",
]
old_existing = next((p for p in candidate_old_human if p.exists()), None)

print("current human pretrain:", human_best.exists(), human_best)
print("old human candidate:", old_existing)

if not human_best.exists() and old_existing is not None:
    shutil.copy2(old_existing, human_best)
    print("✅ 기존 human HuFEP checkpoint를 새 OUTPUT_DIR로 복사했습니다.")
    print("source:", old_existing)
    print("target:", human_best)

if not human_best.exists():
    print("⚠️ 기존 human HuFEP checkpoint가 없어서 archive.zip으로 human pretrain을 시작합니다.")
    human_train_df = human_manifest[human_manifest["split"] == "train"].copy()
    human_val_df = human_manifest[human_manifest["split"] == "val"].copy()

    human_train_ds = SimpleImageDataset(human_train_df, HUMAN_CLASSES, train_tf)
    human_val_ds = SimpleImageDataset(human_val_df, HUMAN_CLASSES, eval_tf)

    human_sampler = None
    if HUMAN_USE_WEIGHTED_SAMPLER:
        human_sampler = make_weighted_sampler(
            human_train_ds.df["label_idx"].astype(int).values,
            len(HUMAN_CLASSES),
            power=HUMAN_SAMPLER_WEIGHT_POWER,
        )

    human_class_weight_tensor = None
    if HUMAN_USE_CLASS_WEIGHT_LOSS:
        human_class_weight_tensor, counts = compute_class_weights(
            human_train_ds.df["label_idx"].astype(int).values,
            len(HUMAN_CLASSES),
            power=HUMAN_CLASS_WEIGHT_POWER,
            clip_max=HUMAN_CLASS_WEIGHT_CLIP_MAX,
        )
        print("✅ human class counts:", {HUMAN_IDX_TO_CLASS[i]: int(c) for i, c in enumerate(counts)})
        print("✅ human class weight:", human_class_weight_tensor.tolist())

    human_train_loader = make_loader(human_train_ds, train=True, sampler=human_sampler)
    human_val_loader = make_loader(human_val_ds, train=False)

    human_model = create_vith14(num_classes=len(HUMAN_CLASSES))
    human_model, human_best_f1 = train_loop(
        human_model,
        human_train_loader,
        human_val_loader,
        HUMAN_PRETRAIN_DIR,
        HUMAN_CLASSES,
        max_epochs=EPOCHS_ARCHIVE_HUMAN_PRETRAIN,
        patience=HUMAN_PRETRAIN_PATIENCE,
        stage_switch_epoch=HUMAN_PRETRAIN_SWITCH_FULL_EPOCH,
        full_after_switch=True,
        class_weight_tensor=human_class_weight_tensor,
        resume=True,
        run_name="Human-HuFEP-Pretrain-ViTH14-336",
    )
    print("✅ human pretrain finished. best_f1:", human_best_f1)
else:
    print("✅ human HuFEP checkpoint 준비 완료:", human_best)

print("HUMAN_PRETRAIN_DIR best:", human_best.exists(), human_best)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

# =========================================================
# 10-1. manifest / crop cache 상태 확인
# =========================================================
import pandas as pd, os

split_csv = OUTPUT_DIR / "dog_manifest_split.csv"
crop_csv = OUTPUT_DIR / "dog_crop_manifest.csv"

if not split_csv.exists():
    split_csv = OLD_OUTPUT_DIR / "dog_manifest_split.csv"
if not crop_csv.exists():
    crop_csv = OLD_OUTPUT_DIR / "dog_crop_manifest.csv"

print("split_csv:", split_csv, split_csv.exists())
print("crop_csv:", crop_csv, crop_csv.exists())

df = pd.read_csv(split_csv)
cf = pd.read_csv(crop_csv)

print("=== dog_manifest_split split count ===")
print(df["split"].value_counts(dropna=False))

print("\n=== dog_manifest_split label x split ===")
print(pd.crosstab(df["label"], df["split"]))

print("\n=== dog_crop_manifest split count ===")
print(cf["split"].value_counts(dropna=False))

print("\n=== detected count ===")
print(cf["detected"].value_counts(dropna=False))

print("\n=== crop file exists check ===")
for col in ALL_DOG_CROP_COLS_5:
    if col in cf.columns:
        exists_count = cf[col].apply(lambda p: os.path.exists(str(p))).sum()
        print(col, "exists:", exists_count, "/", len(cf))


In [ ]:

# =========================================================
# 11. HuFEP ViT-H/14 336 학습 — 5CROP SIMULTANEOUS
# - 논문처럼 Human Facial Expression Pre-trained 모델을 강아지 감정으로 추가 학습
# - 이미지 1장당 5개 crop을 동시에 넣고 weighted logits 평균으로 loss 계산
# - dataset 길이는 train 원본 이미지 수 기준으로 유지
# =========================================================

from pathlib import Path
import math, json, gc
import torch
import torch.nn.functional as F
import timm
from torch.utils.data import WeightedRandomSampler

# ---------------------------------------------------------
# 0) 필수 함수/변수 확인
# ---------------------------------------------------------
assert "DogCropDataset" in globals(), "8번 Dataset 칸을 먼저 실행해야 합니다."
assert "MultiCropSimultaneousDogDataset" in globals(), "8번 MultiCropSimultaneousDogDataset 정의를 먼저 실행해야 합니다."
assert "make_loader" in globals(), "8번 make_loader 정의를 먼저 실행해야 합니다."
assert "train_hufep_stable_loop" in globals(), "10-X HuFEP stable 학습 함수 칸을 먼저 실행해야 합니다."
assert "dog_crop_manifest" in globals(), "7번 crop manifest 생성/로드 칸을 먼저 실행해야 합니다."
assert USE_5CROP_SIMULTANEOUS_TRAIN is True

# ---------------------------------------------------------
# 1) HuFEP crop 설정: 5개 동시 입력
# ---------------------------------------------------------
HUFEP_RANDOM_ONE_CROP_PER_IMAGE = False
HUFEP_VITH14_DOG_TRAIN_CROP_COLS = ALL_DOG_CROP_COLS_5[:]
HUFEP_DETECTED_TRAIN_CROP_COLS = HUFEP_VITH14_DOG_TRAIN_CROP_COLS[:]

print("✅ HuFEP train mode: 5CROP SIMULTANEOUS")
print("✅ HuFEP train crops:", HUFEP_VITH14_DOG_TRAIN_CROP_COLS)
print("✅ crop weights:", CROP_WEIGHTS)

# ---------------------------------------------------------
# 2) train / val dataframe 준비
# ---------------------------------------------------------
train_df = dog_crop_manifest[dog_crop_manifest["split"] == "train"].copy()
val_df = dog_crop_manifest[dog_crop_manifest["split"] == "val"].copy()

print("train_df:", len(train_df))
print("val_df:", len(val_df))
assert len(train_df) > 0, "train split 데이터가 없습니다."
assert len(val_df) > 0, "val split 데이터가 없습니다."

# ---------------------------------------------------------
# 3) HuFEP train/val dataset
# ---------------------------------------------------------
hufep_train_ds = MultiCropSimultaneousDogDataset(
    train_df,
    HUFEP_VITH14_DOG_TRAIN_CROP_COLS,
    train_tf,
    detected_only=HUFEP_DETECTED_ONLY_TRAIN,
    min_conf=HUFEP_MIN_DET_CONF,
    require_all=False,
)

# val도 5crop simultaneous로 평가해서 best 저장 기준을 최종 추론 구조와 맞춥니다.
hufep_val_ds = MultiCropSimultaneousDogDataset(
    val_df,
    HUFEP_VITH14_DOG_TRAIN_CROP_COLS,
    eval_tf,
    detected_only=False,
    min_conf=0.0,
    require_all=False,
)

print("✅ HuFEP train images:", len(hufep_train_ds), "| crops/image:", len(HUFEP_VITH14_DOG_TRAIN_CROP_COLS))
print("✅ HuFEP val images:", len(hufep_val_ds), "| crops/image:", len(HUFEP_VITH14_DOG_TRAIN_CROP_COLS))
assert len(hufep_train_ds) > 0
assert len(hufep_val_ds) > 0

# ---------------------------------------------------------
# 4) weighted sampler 준비
# ---------------------------------------------------------
def build_weighted_sampler_from_dataset(ds):
    labels = ds.df["label_idx"].astype(int).values
    counts = {}
    for y in labels:
        counts[int(y)] = counts.get(int(y), 0) + 1
    weights = [1.0 / max(1, counts[int(y)]) for y in labels]
    print("✅ HuFEP train class counts:", {DOG_IDX_TO_CLASS[k]: v for k, v in counts.items()})
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(weights),
        num_samples=len(weights),
        replacement=True,
    )

hufep_sampler = build_weighted_sampler_from_dataset(hufep_train_ds) if HUFEP_USE_WEIGHTED_SAMPLER else None
hufep_train_loader = make_loader(hufep_train_ds, train=True, sampler=hufep_sampler)
hufep_val_loader = make_loader(hufep_val_ds, train=False)

print("✅ HuFEP train loader batches:", len(hufep_train_loader))
print("✅ HuFEP val loader batches:", len(hufep_val_loader))

# ---------------------------------------------------------
# 5) human HuFEP checkpoint 찾기
# ---------------------------------------------------------
old_human_best = None
for p in [
    OLD_HUMAN_PRETRAIN_DIR / "best_model.pt",
    HUMAN_PRETRAIN_DIR / "best_model.pt",
    OLD_OUTPUT_DIR / "00_archive_HuFEP_ViTH14_human_pretrain" / "best_model.pt",
]:
    if p.exists():
        old_human_best = p
        break

assert old_human_best is not None and old_human_best.exists(), "human HuFEP best_model.pt를 찾을 수 없습니다. 0~10번 칸 또는 기존 224 HuFEP pretrain 결과를 확인하세요."
print("✅ using human HuFEP checkpoint:", old_human_best)

# ---------------------------------------------------------
# 6) 현재 IMG_SIZE에 맞게 position embedding 보간
# ---------------------------------------------------------
def resize_pos_embed_to_current_img_size(state_dict, model):
    if "pos_embed" not in state_dict or not hasattr(model, "pos_embed"):
        return state_dict
    old_pos = state_dict["pos_embed"]
    new_pos = model.pos_embed
    if old_pos.shape == new_pos.shape:
        print("✅ pos_embed shape 동일:", tuple(old_pos.shape))
        return state_dict

    print("🔁 resize pos_embed:", tuple(old_pos.shape), "->", tuple(new_pos.shape))
    num_prefix = 1
    old_prefix = old_pos[:, :num_prefix]
    old_grid = old_pos[:, num_prefix:]
    new_grid = new_pos[:, num_prefix:]
    old_size = int(math.sqrt(old_grid.shape[1]))
    new_size = int(math.sqrt(new_grid.shape[1]))
    assert old_size * old_size == old_grid.shape[1], f"old pos_embed grid가 정사각형이 아닙니다: {old_grid.shape}"
    assert new_size * new_size == new_grid.shape[1], f"new pos_embed grid가 정사각형이 아닙니다: {new_grid.shape}"

    old_grid = old_grid.reshape(1, old_size, old_size, -1).permute(0, 3, 1, 2)
    new_grid_resized = F.interpolate(old_grid, size=(new_size, new_size), mode="bicubic", align_corners=False)
    new_grid_resized = new_grid_resized.permute(0, 2, 3, 1).reshape(1, new_size * new_size, -1)
    state_dict = dict(state_dict)
    state_dict["pos_embed"] = torch.cat([old_prefix, new_grid_resized], dim=1)
    print("✅ resized pos_embed:", tuple(state_dict["pos_embed"].shape))
    return state_dict

# ---------------------------------------------------------
# 7) HuFEP dog 모델 생성
# ---------------------------------------------------------
hufep_dog_model = timm.create_model(
    VITH14_MODEL_NAME,
    pretrained=False,
    num_classes=len(DOG_CLASSES),
    img_size=IMG_SIZE,
)

if hasattr(hufep_dog_model, "set_grad_checkpointing"):
    hufep_dog_model.set_grad_checkpointing(True)
    print("✅ grad checkpointing enabled for HuFEP")

# ---------------------------------------------------------
# 8) human HuFEP backbone 로드
# ---------------------------------------------------------
human_ckpt = torch.load(old_human_best, map_location="cpu")
sd = human_ckpt["model"] if isinstance(human_ckpt, dict) and "model" in human_ckpt else human_ckpt

target_sd = hufep_dog_model.state_dict()
filtered = {}
for k, v in sd.items():
    if k.startswith("head.") or k.startswith("fc.") or k.startswith("classifier."):
        continue
    if k == "pos_embed":
        filtered[k] = v
        continue
    if k in target_sd and target_sd[k].shape == v.shape:
        filtered[k] = v

filtered = resize_pos_embed_to_current_img_size(filtered, hufep_dog_model)
msg = hufep_dog_model.load_state_dict(filtered, strict=False)
print("✅ loaded human HuFEP backbone into dog HuFEP")
print("missing keys:", len(msg.missing_keys), "unexpected:", len(msg.unexpected_keys))

# ---------------------------------------------------------
# 9) HuFEP stable 학습 실행
# ---------------------------------------------------------
HUFEP_DOG_STABLE_DIR.mkdir(parents=True, exist_ok=True)
print("✅ HuFEP output dir:", HUFEP_DOG_STABLE_DIR)

hufep_dog_model, hufep_best = train_hufep_stable_loop(
    hufep_dog_model,
    hufep_train_loader,
    hufep_val_loader,
    HUFEP_DOG_STABLE_DIR,
    DOG_CLASSES,
    max_epochs=MAX_EPOCHS_DOG_HUFEP,
    patience=EARLY_STOP_PATIENCE_DOG_HUFEP,
    resume=True,
    run_name="HuFEP-Archive-ViTH14-DogFace-336-5CROP-SIMUL",
)

print("\n==============================")
print("✅ HuFEP 336 5CROP SIMULTANEOUS 학습 완료")
print("==============================")
print("best macro_f1:", hufep_best)
print("best model:", HUFEP_DOG_STABLE_DIR / "best_model.pt")
print("==============================")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

# =========================================================
# 12. baseline ViT-H/14 dog-only 학습 — 5CROP SIMULTANEOUS
# - 논문의 baseline 역할
# - HuFEP 없이 강아지 감정 데이터만으로 학습
# =========================================================
assert "MultiCropSimultaneousDogDataset" in globals(), "8번 MultiCropSimultaneousDogDataset 정의를 먼저 실행해야 합니다."
assert USE_5CROP_SIMULTANEOUS_TRAIN is True

dog_train_df = dog_crop_manifest[dog_crop_manifest["split"] == "train"].copy()
dog_val_df = dog_crop_manifest[dog_crop_manifest["split"] == "val"].copy()

TIMM_TRAIN_CROP_COLS = ALL_DOG_CROP_COLS_5[:]

baseline_train_ds = MultiCropSimultaneousDogDataset(
    dog_train_df,
    TIMM_TRAIN_CROP_COLS,
    train_tf,
    detected_only=False,
    require_all=False,
)

baseline_val_ds = MultiCropSimultaneousDogDataset(
    dog_val_df,
    TIMM_TRAIN_CROP_COLS,
    eval_tf,
    detected_only=False,
    require_all=False,
)

print("✅ baseline train images:", len(baseline_train_ds), "| crops/image:", len(TIMM_TRAIN_CROP_COLS))
print("✅ baseline val images:", len(baseline_val_ds), "| crops/image:", len(TIMM_TRAIN_CROP_COLS))

baseline_train_loader = make_loader(baseline_train_ds, train=True)
baseline_val_loader = make_loader(baseline_val_ds, train=False)

baseline_model = create_vith14(num_classes=len(DOG_CLASSES))
baseline_model, baseline_best = train_loop(
    baseline_model,
    baseline_train_loader,
    baseline_val_loader,
    BASELINE_DIR,
    DOG_CLASSES,
    max_epochs=MAX_EPOCHS_DOG,
    patience=EARLY_STOP_PATIENCE_DOG,
    stage_switch_epoch=5,
    full_after_switch=True,
    class_weight_tensor=None,
    resume=True,
    run_name="Baseline-ViTH14-DogOnly-336-5CROP-SIMUL",
)

print("✅ baseline dog model:", BASELINE_DIR / "best_model.pt")


In [ ]:
# =========================================================
# 12-1. 필수 모델 확인
# =========================================================
required = {
    "00 human HuFEP pretrain": HUMAN_PRETRAIN_DIR / "best_model.pt",
    "01 baseline dog-only": BASELINE_DIR / "best_model.pt",
    "02 final HuFEP dog": HUFEP_DOG_DIR / "best_model.pt",
}

for k, p in required.items():
    print(f"{k:28s}: {'✅' if p.exists() else '❌'} {p}")


In [ ]:

# =========================================================
# 13. HuFEP + baseline ensemble weight 탐색
# - 336 5crop simultaneous HuFEP 우선 사용
# - 없으면 기존 224 HuFEP fallback 가능
# - HuFEP crop 모드도 validation에서 자동 비교
# =========================================================
base_path = BASELINE_DIR / "best_model.pt"

stable_hufep_path = HUFEP_DOG_STABLE_DIR / "best_model.pt"
old_hufep_path = OLD_HUFEP_DOG_DIR / "best_model.pt"

if stable_hufep_path.exists():
    hufep_path = stable_hufep_path
    hufep_source = "336_5crop_simul"
elif old_hufep_path.exists():
    hufep_path = old_hufep_path
    hufep_source = "old_224_fallback"
else:
    raise FileNotFoundError("HuFEP best_model.pt가 없습니다.")

assert base_path.exists(), base_path
assert hufep_path.exists(), hufep_path

print("✅ baseline model:", base_path)
print("✅ HuFEP model:", hufep_path)
print("✅ HuFEP source:", hufep_source)

baseline_eval_model = load_vith14_checkpoint(base_path, len(DOG_CLASSES))
hufep_eval_model = load_vith14_checkpoint(hufep_path, len(DOG_CLASSES))

val_df = dog_crop_manifest[dog_crop_manifest["split"] == "val"].copy()

def existing_crop_cols(df, cols):
    out = []
    for c in cols:
        if c in df.columns and df[c].apply(lambda x: isinstance(x, str) and Path(x).exists()).any():
            out.append(c)
    if not out:
        raise RuntimeError(f"사용 가능한 crop column이 없습니다: {cols}")
    return out

baseline_crop_cols = existing_crop_cols(val_df, ["crop_headsafe", "crop_tight", "crop_yolo", "crop_center", "crop_original"])
base_probs, y_true = predict_probs_for_df(baseline_eval_model, val_df, baseline_crop_cols, DOG_CLASSES)
base_m = metrics_from_probs(base_probs, y_true, DOG_CLASSES)
print("baseline weighted-5crop val:", {k: v for k, v in base_m.items() if k != "pred"})

hufep_crop_candidates = {
    "weighted5_all": ["crop_headsafe", "crop_tight", "crop_yolo", "crop_center", "crop_original"],
    "best3_head_tight_yolo": ["crop_headsafe", "crop_tight", "crop_yolo"],
    "face_head": ["crop_headsafe", "crop_tight"],
    "head_yolo": ["crop_headsafe", "crop_yolo"],
    "head_only": ["crop_headsafe"],
}

best = None
all_results = []
for crop_name, cols_raw in hufep_crop_candidates.items():
    hufep_cols = existing_crop_cols(val_df, cols_raw)
    print("\nTesting HuFEP crop mode:", crop_name, hufep_cols)
    hufep_probs, y_true2 = predict_probs_for_df(hufep_eval_model, val_df, hufep_cols, DOG_CLASSES)
    assert np.array_equal(y_true, y_true2)

    hufep_m = metrics_from_probs(hufep_probs, y_true, DOG_CLASSES)
    print("hufep weighted-crop val:", {k: v for k, v in hufep_m.items() if k != "pred"})

    all_results.append({
        "type": "single_hufep",
        "hufep_crop_name": crop_name,
        "hufep_crops": hufep_cols,
        "hufep_source": hufep_source,
        "hufep_model_path": str(hufep_path),
        **{k: float(v) for k, v in hufep_m.items() if k != "pred"},
    })

    for w in np.arange(0.0, 1.0001, ENSEMBLE_STEP):
        probs = w * base_probs + (1.0 - w) * hufep_probs
        m = metrics_from_probs(probs, y_true, DOG_CLASSES)
        item = {
            "type": "baseline_hufep_ensemble",
            "w_baseline": float(w),
            "w_hufep": float(1.0 - w),
            "baseline_crops": baseline_crop_cols,
            "hufep_crop_name": crop_name,
            "hufep_crops": hufep_cols,
            "baseline_model_path": str(base_path),
            "hufep_model_path": str(hufep_path),
            "hufep_source": hufep_source,
            **{k: float(v) for k, v in m.items() if k != "pred"},
        }
        all_results.append(item)
        if best is None or item["macro_f1"] > best["macro_f1"]:
            best = item

print("\n🔥 best ensemble val:")
print(best)
with open(OUTPUT_DIR / "best_ensemble_config.json", "w", encoding="utf-8") as f:
    json.dump(best, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "ensemble_val_all_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print("✅ saved:", OUTPUT_DIR / "best_ensemble_config.json")
print("✅ saved:", OUTPUT_DIR / "ensemble_val_all_results.json")


In [ ]:

# =========================================================
# 14. test 최종 평가
# - 13번 best_ensemble_config.json의 모델 경로/crop 설정/가중 crop 평균 그대로 사용
# =========================================================
with open(OUTPUT_DIR / "best_ensemble_config.json", "r", encoding="utf-8") as f:
    ens_cfg = json.load(f)

print("✅ loaded ensemble config:")
print(json.dumps(ens_cfg, ensure_ascii=False, indent=2))

base_path = Path(ens_cfg["baseline_model_path"])
hufep_path = Path(ens_cfg["hufep_model_path"])
assert base_path.exists(), base_path
assert hufep_path.exists(), hufep_path

baseline_eval_model = load_vith14_checkpoint(base_path, len(DOG_CLASSES))
hufep_eval_model = load_vith14_checkpoint(hufep_path, len(DOG_CLASSES))

test_df = dog_crop_manifest[dog_crop_manifest["split"] == "test"].copy()

base_probs_test, y_test = predict_probs_for_df(
    baseline_eval_model,
    test_df,
    ens_cfg["baseline_crops"],
    DOG_CLASSES,
)

hufep_probs_test, y_test2 = predict_probs_for_df(
    hufep_eval_model,
    test_df,
    ens_cfg["hufep_crops"],
    DOG_CLASSES,
)
assert np.array_equal(y_test, y_test2)

base_test = metrics_from_probs(base_probs_test, y_test, DOG_CLASSES)
hufep_test = metrics_from_probs(hufep_probs_test, y_test, DOG_CLASSES)

w = float(ens_cfg["w_baseline"])
ens_probs = w * base_probs_test + (1.0 - w) * hufep_probs_test
ens_test = metrics_from_probs(ens_probs, y_test, DOG_CLASSES)

print("baseline test:", {k: v for k, v in base_test.items() if k != "pred"})
print("HuFEP test:", {k: v for k, v in hufep_test.items() if k != "pred"})
print("ensemble test:", {k: v for k, v in ens_test.items() if k != "pred"})

final_report = {
    "baseline_test": {k: float(v) for k, v in base_test.items() if k != "pred"},
    "hufep_test": {k: float(v) for k, v in hufep_test.items() if k != "pred"},
    "ensemble_test": {k: float(v) for k, v in ens_test.items() if k != "pred"},
    "ensemble_config": ens_cfg,
    "classes": DOG_CLASSES,
}
with open(OUTPUT_DIR / "test_report_final.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

save_reports(OUTPUT_DIR, DOG_CLASSES, y_test, ens_test["pred"], prefix="test_ensemble")
print("✅ saved:", OUTPUT_DIR / "test_report_final.json")


In [ ]:
# =========================================================
# 15. 설정 저장
# =========================================================
config = {
    "PROJECT_BASE": str(PROJECT_BASE),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "VITH14_MODEL_NAME": VITH14_MODEL_NAME,
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "MAX_EPOCHS_DOG": MAX_EPOCHS_DOG,
    "EARLY_STOP_PATIENCE_DOG": EARLY_STOP_PATIENCE_DOG,
    "EPOCHS_ARCHIVE_HUMAN_PRETRAIN": EPOCHS_ARCHIVE_HUMAN_PRETRAIN,
    "HUMAN_PRETRAIN_PATIENCE": HUMAN_PRETRAIN_PATIENCE,
    "LR_HEAD": LR_HEAD,
    "LR_BACKBONE": LR_BACKBONE,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "LABEL_SMOOTHING": LABEL_SMOOTHING,
    "DETECT_IMGSZ": DETECT_IMGSZ,
    "VAL_RATIO": VAL_RATIO,
    "TEST_RATIO": TEST_RATIO,
    "HUMAN_USE_WEIGHTED_SAMPLER": HUMAN_USE_WEIGHTED_SAMPLER,
    "HUMAN_USE_CLASS_WEIGHT_LOSS": HUMAN_USE_CLASS_WEIGHT_LOSS,
    "HUFEP_VITH14_DOG_TRAIN_CROP_COLS": HUFEP_VITH14_DOG_TRAIN_CROP_COLS,
    "TIMM_TRAIN_CROP_COLS": TIMM_TRAIN_CROP_COLS,
    "HUFEP_RANDOM_ONE_CROP_PER_IMAGE": HUFEP_RANDOM_ONE_CROP_PER_IMAGE,
    "USE_5CROP_SIMULTANEOUS_TRAIN": USE_5CROP_SIMULTANEOUS_TRAIN,
    "USE_5CROP_SIMULTANEOUS_VAL": USE_5CROP_SIMULTANEOUS_VAL,
    "CROP_WEIGHTS": CROP_WEIGHTS,
    "MULTICROP_WEIGHT_LIST": MULTICROP_WEIGHT_LIST,
}
with open(OUTPUT_DIR / "training_config_final.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("✅ saved config:", OUTPUT_DIR / "training_config_final.json")


In [ ]:
!find /content -name "*.pt"

/content/drive/MyDrive/best_emotion_model (7).pt
/content/drive/MyDrive/best_emotion_model.pt
/content/drive/MyDrive/pet-ai2/output_emotion_v2/best_emotion_model.pt
/content/drive/MyDrive/pet-ai2/output_emotion_hybrid_v2/best_emotion_model.pt


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/best_emotion_model.pt')

FileNotFoundError: Cannot find file: /content/drive/MyDrive/best_emotion_model.pt